In [1]:
%load_ext autoreload
%autoreload 2

# Recoil heating comparison

We consider a prolate silica spheroid centered at the origin, with its long axis aligned with the $x$ axis. The particle is illuminated by a monochromatic plane wave propagating along $+z$ and linearly polarized along $x$. Its dimensions and illumination parameters are defined below.

In [2]:
from IPython.display import display, Markdown, Math, HTML

import numpy as np
from scipy.constants import epsilon_0, c, hbar
from pyGDM2 import fields

from optomechanics_levitation import particle_properties, rotate_particle, evaluate_incident_field, evaluate_internal_field, evaluate_far_field

In [3]:
## PARTICLE PROPERTIES
long_axis = 100.0; long_radius = long_axis / 2.0; aspect_ratio = 2.0; short_radius = long_radius / aspect_ratio # [nm]
wavelength = 1550.0 # [nm]
intensity = 1.0e9 # [W/m^2]

# Discretization step
if aspect_ratio == 2.0:
    mesh_divisions = 35.0
else:
    mesh_divisions = 65.0
step = long_axis / mesh_divisions # [nm]

# x-oriented ellipsoidal particle
ellipsoidal_particle = particle_properties(particle_type="spheroid", material_name="sio2", step_nm=step, mesh="cube",
                                           center=True, R1=long_radius / step - 0.2, R2=short_radius / step - 0.2, R3=short_radius / step - 0.2)

# INTERACTION PARAMETERS
far_field_radius = 1e6 # [nm]

## Militaru's approach

Following [Militaru's treatment](https://doi.org/10.3929/ethz-b-000598122), we describe the spheroid in the Rayleigh limit as an anisotropic electric dipole. Its semiaxes are $a$ along $x$ and $b$ along $y$ and $z$. The polarizability along each axis is

$$
\alpha_i=\epsilon_0 V\frac{\epsilon_r-1}{1+L_i(\epsilon_r-1)},
\qquad V=\frac{4\pi}{3}ab^2,
$$

where $L_i$ is the corresponding depolarization factor and $\epsilon_r$ is the real part of silica's relative permittivity at the illumination wavelength.

Because the incident field is polarized along $x$, the scattered power depends on $\alpha_x$. We obtain the photon-scattering rate from

$$
P_{\mathrm{sc}}=\frac{\omega^4\alpha_x^2E_0^2}{12\pi\epsilon_0c^3},
\qquad
\Gamma_{\mathrm{sc}}=\frac{P_{\mathrm{sc}}}{\hbar\omega},
$$

with $E_0=\sqrt{2I/(\epsilon_0c)}$ and $\omega=2\pi c/\lambda$. Finally, the polarizability anisotropy $\Delta\alpha=\alpha_x-\alpha_y$ gives the librational torque-noise prediction used in our comparison:

$$
S_{\tau\tau}^{\mathrm{M}}
=\frac{\hbar^2\Gamma_{\mathrm{sc}}}{2\pi}
\left(\frac{\Delta\alpha}{\alpha_x}\right)^2.
$$

The spheroid is symmetric about its long axis, so this expression gives the same prediction for librations about $y$ and $z$.

In [4]:
def militaru_torque_recoil_heating(long_radius_nm, short_radius_nm, epsilon_r, intensity, wavelength):
    a = long_radius_nm * 1e-9; b = short_radius_nm * 1e-9
    e = np.sqrt(1 - (b / a)**2)

    L_x = (1 - e**2) / (2 * e**3) * (np.log((1 + e) / (1 - e)) - 2 * e)
    L_y = (1 - L_x) / 2
    volume = 4 * np.pi * a * b**2 / 3

    alpha_x = epsilon_0 * volume * (epsilon_r - 1) / (1 + L_x * (epsilon_r - 1))
    alpha_y = epsilon_0 * volume * (epsilon_r - 1) / (1 + L_y * (epsilon_r - 1))
    delta_alpha = alpha_x - alpha_y

    E0 = np.sqrt(2 * intensity / (epsilon_0 * c))
    omega0 = 2 * np.pi * c / (wavelength * 1e-9)

    P_sc = omega0**4 * alpha_x**2 * E0**2 / (12 * np.pi * epsilon_0 * c**3)
    Gamma_sc = P_sc / (hbar * omega0)
    S_tau_tau = hbar**2 * Gamma_sc / (2 * np.pi) * (delta_alpha / alpha_x)**2

    return Gamma_sc, S_tau_tau

gamma_m_sc, s_m_tau_tau = militaru_torque_recoil_heating(long_radius, short_radius, ellipsoidal_particle["material"].epsilon(wavelength).real, intensity, wavelength)
d_militaru = 2 * np.pi * s_m_tau_tau / (hbar**2 * gamma_m_sc)

## Simulator approach

We represent the spheroid with a cubic mesh of dipoles. For each particle orientation, we evaluate the incident plane wave at the dipole positions, solve for the internal electric field, and calculate the scattered electric field $\mathbf E_{\mathrm{sc}}$ on a sphere in the far field. Integrating its intensity over all observation directions gives the photon-scattering rate:

$$
\Gamma_{\mathrm{sc}}^{\mathrm{sim}}
=\frac{\epsilon_0cr^2}{2\hbar\omega}
\int_{4\pi}|\mathbf E_{\mathrm{sc}}|^2\,d\Omega.
$$

To evaluate librations about $j\in\{y,z\}$, we rotate the particle by small angles $+\delta$ and $-\delta$ about its center and recalculate the scattered field. Its orientation derivative is approximated by

$$
\partial_j\mathbf E_{\mathrm{sc}}
\approx
\frac{\mathbf E_{\mathrm{sc}}(+\delta)-\mathbf E_{\mathrm{sc}}(-\delta)}
{2\delta},
\qquad \delta=0.005\ \mathrm{rad}.
$$

We then integrate the squared derivative over the same observation sphere. The normalized rotational response and torque-noise density are

$$
D_j=
\frac{\int_{4\pi}|\partial_j\mathbf E_{\mathrm{sc}}|^2\,d\Omega}
     {\int_{4\pi}|\mathbf E_{\mathrm{sc}}|^2\,d\Omega},
\qquad
S_{\tau_j\tau_j}^{\mathrm{sim}}
=\frac{\hbar^2\Gamma_{\mathrm{sc}}^{\mathrm{sim}}}{2\pi}D_j.
$$

In [5]:
def scattering_rate(scattered_field, theta, phi, radius_nm, wavelength_nm):
    field_squared = np.sum(np.abs(scattered_field)**2, axis=1)
    field_squared = field_squared.reshape(len(theta), len(phi))

    dtheta = theta[1] - theta[0]
    dphi = phi[1] - phi[0]
    integral = np.sum(field_squared * np.sin(theta)[:, None]) * dtheta * dphi

    radius_m = radius_nm * 1e-9
    omega = 2 * np.pi * c / (wavelength_nm * 1e-9)
    P_sc = epsilon_0 * c * radius_m**2 * integral / 2

    return P_sc / (hbar * omega)

def simulation_recoil_heating(ellipsoidal_particle, intensity, wavelength, far_field_distance, rotation_axes=("y", "z"), angle_rad=0.005):
    E0 = np.sqrt(2 * intensity / (epsilon_0 * c))
    environment = {"eps1": 1.0, "eps2": 1.0, "eps3": 1.0, "spacing": 0.0}

    theta = (np.arange(64) + 0.5) * np.pi / 64; phi = np.linspace(0, 2 * np.pi, 64, endpoint=False)
    theta_grid, phi_grid = np.meshgrid(theta, phi, indexing="ij")
    directions = np.column_stack(((np.sin(theta_grid) * np.cos(phi_grid)).ravel(), (np.sin(theta_grid) * np.sin(phi_grid)).ravel(), np.cos(theta_grid).ravel()))
    observation_points = far_field_distance * directions

    geometry = ellipsoidal_particle["geometry_nm"]
    rotation_origin = geometry.mean(axis=0)
    def scattered_field_at(axis=None, rotation_rad=0.0):
        particle = ellipsoidal_particle.copy()

        if axis is not None:
            angles_deg = np.zeros(3)
            angles_deg["xyz".index(axis)] = np.degrees(rotation_rad)
            particle["geometry_nm"] = rotate_particle(geometry, angles_deg, origin_nm=rotation_origin)

        incident_field = evaluate_incident_field(field_generator=fields.plane_wave, positions_nm=particle["geometry_nm"], 
                                                 wavelength_nm=wavelength, environment=environment, amplitude_v_per_m=E0, field="E", 
                                                 inc_angle=0.0, inc_plane="xz", E_p=1.0, E_s=0.0)
        interaction = evaluate_internal_field(particle=particle, incident_electric_field=incident_field, wavelength_nm=wavelength, environment=environment)
        
        return evaluate_far_field(interaction_result=interaction, positions_nm=observation_points, field="E")

    scattered_field = scattered_field_at()
    Gamma_sc = scattering_rate(scattered_field, theta, phi, far_field_distance, wavelength)

    D = {}
    S_tau_tau = {}
    for axis in rotation_axes:
        field_plus = scattered_field_at(axis, +angle_rad)
        field_minus = scattered_field_at(axis, -angle_rad)
        field_derivative = (field_plus - field_minus) / (2 * angle_rad)

        gamma_derivative = scattering_rate(field_derivative, theta, phi, far_field_distance, wavelength)
        D[axis] = gamma_derivative / Gamma_sc
        S_tau_tau[axis] = hbar**2 * gamma_derivative / (2 * np.pi)

    return Gamma_sc, D, S_tau_tau

gamma_sim_sc, d_sim, s_sim_tau_tau = simulation_recoil_heating(ellipsoidal_particle, intensity, wavelength, far_field_radius, rotation_axes=("y", "z"))

# Results

The table compares Militaru's analytical prediction with the numerical results for the photon-scattering rate $\Gamma_{\mathrm{sc}}$, the torque-noise spectral densities $S_{\tau_y\tau_y}$ and $S_{\tau_z\tau_z}$, and the dimensionless rotational responses $D_y$ and $D_z$. The final column gives the absolute relative difference with respect to the analytical prediction.

In [6]:
## RELATIVE DIFFERENCES
gamma_sc_diff = np.abs(gamma_sim_sc - gamma_m_sc) / gamma_m_sc
s_tau_tau_diff = {axis: np.abs(s_sim_tau_tau[axis] - s_m_tau_tau) / s_m_tau_tau for axis in ("y", "z")}
d_diff = {axis: np.abs(d_sim[axis] - d_militaru) / d_militaru for axis in ("y", "z")}

In [7]:
def sci_notation(value, decimals):
    """ Function to write values in scientific notation """
    mantissa, exponent = f"{value:.{decimals}e}".split("e")
    return rf"{mantissa} \times 10^{{{int(exponent)}}}" 

militaru_benchmark_table = rf"""

| Observable | Militaru | Simulation | Relative difference |
|:-:|:-:|:-:|:-:|
| $\Gamma_{{sc}}\,[\mathrm{{s}}^{{-1}}]$ | ${sci_notation(gamma_m_sc, 3)}$ | ${sci_notation(gamma_sim_sc, 3)}$ | ${100 * gamma_sc_diff:.2f}\%$ |
| $S_{{\tau_y\tau_y}}\,[\mathrm{{N}}^{{2}}\mathrm{{m}}^{{2}}/\mathrm{{Hz}}]$ | ${sci_notation(s_m_tau_tau, 3)}$ | ${sci_notation(s_sim_tau_tau['y'], 3)}$ | ${100 * s_tau_tau_diff['y']:.2f}\%$ |
| $S_{{\tau_z\tau_z}}\,[\mathrm{{N}}^{{2}}\mathrm{{m}}^{{2}}/\mathrm{{Hz}}]$ | ${sci_notation(s_m_tau_tau, 3)}$ | ${sci_notation(s_sim_tau_tau['z'], 3)}$ | ${100 * s_tau_tau_diff['z']:.2f}\%$ |
| $D_y$ | ${d_militaru:.6f}$ | ${d_sim['y']:.6f}$ | ${100 * d_diff['y']:.2f}\%$ |
| $D_z$ | ${d_militaru:.6f}$ | ${d_sim['z']:.6f}$ | ${100 * d_diff['z']:.2f}\%$ |

"""

display(Markdown(militaru_benchmark_table))



| Observable | Militaru | Simulation | Relative difference |
|:-:|:-:|:-:|:-:|
| $\Gamma_{sc}\,[\mathrm{s}^{-1}]$ | $1.597 \times 10^{9}$ | $1.613 \times 10^{9}$ | $1.00\%$ |
| $S_{\tau_y\tau_y}\,[\mathrm{N}^{2}\mathrm{m}^{2}/\mathrm{Hz}]$ | $9.114 \times 10^{-62}$ | $9.165 \times 10^{-62}$ | $0.56\%$ |
| $S_{\tau_z\tau_z}\,[\mathrm{N}^{2}\mathrm{m}^{2}/\mathrm{Hz}]$ | $9.114 \times 10^{-62}$ | $9.168 \times 10^{-62}$ | $0.59\%$ |
| $D_y$ | $0.032240$ | $0.032100$ | $0.43\%$ |
| $D_z$ | $0.032240$ | $0.032109$ | $0.40\%$ |



# Notes

**Mesh resolution.** The cubic mesh uses a step size $s=L/N$, where $L$ is the long-axis length and $N$ is `mesh_divisions`. We tested several values of $N$ for the 2:1 spheroid. Refining the mesh improved the representation of the particle, but also increased computation time substantially. We use $N=35$ for this comparison. For a 4:1 spheroid, $N=65$ keeps a similar resolution across the shorter radius.

**Spheroid boundary.** The `pyGDM2.structures.spheroid` generator adds $0.2$ mesh cells to each input radius when selecting dipoles. We therefore pass $r_i/s-0.2$ for each axis so that its selection criterion uses the intended radii. This offset is fixed in mesh cells and does not change with the aspect ratio.

For the 100 nm, 2:1 particle at $N=35$, the difference between the mesh volume, $N_{\mathrm{dipoles}}s^3$, and the target ellipsoid volume decreases from **+4.82% to +0.26%** after this correction. The resulting numerical differences from Militaru's prediction are **1.00%** for $\Gamma_{\mathrm{sc}}$ and **0.56–0.59%** for $S_{\tau\tau}$.